# Train CARL Ratio Estimator (Physics Parameters Only)

Train a parameterized classifier to estimate $r(x|\theta) = p(x|\theta) / p(x|\theta_{\text{SM}})$
using the CARL (Classification-Aware Ratio Learning) method.

CARL uses a pure cross-entropy loss — no score augmentation — making it simpler
and more robust than ALICES, at the cost of slightly less sample efficiency.

**Input**: `data/lhe_data_semi_parametric_b1.h5`  
**Output**: Trained model saved to `models/carl`

## 0. Setup

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt

from madminer.sampling import SampleAugmenter
from madminer.ml import ParameterizedRatioEstimator

logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.INFO,
)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

## 1. Extract Training Data

CARL only needs $(x, \theta_0, y)$ — no joint likelihood ratio or score.
We still extract `r_xz` for validation purposes.

In [ ]:
sampler = SampleAugmenter("data/lhe_data_semi_parametric_b1.h5")

x, theta0, theta1, y, r_xz, t_xz, n_eff = sampler.sample_train_ratio(
    theta0=("random_morphing_points", (500, [
        ("flat", -20.0, 20.0),
        ("flat", -20.0, 20.0),
    ])),
    theta1=("benchmark", "sm"),
    n_samples=500_000,
    folder="data/samples_carl",
    filename="train",
    nuisance_score=False,
    test_split=0.2,
    validation_split=0.2,
    partition="train",
)

print(f"Training: x={x.shape}, theta0={theta0.shape}, y={y.shape}")
print(f"n_eff: {n_eff:.0f}")

# Free memory
del x, theta0, theta1, y, r_xz, t_xz

In [ ]:
x_test, theta0_test, theta1_test, y_test, r_xz_test, _, _ = sampler.sample_train_ratio(
    theta0=("random_morphing_points", (200, [
        ("flat", -20.0, 20.0),
        ("flat", -20.0, 20.0),
    ])),
    theta1=("benchmark", "sm"),
    n_samples=100_000,
    folder="data/samples_carl",
    filename="test",
    nuisance_score=False,
    test_split=0.2,
    validation_split=0.2,
    partition="test",
)

print(f"Test: {x_test.shape[0]} samples")

# Keep r_xz_test for validation, free the rest
del x_test, theta0_test, theta1_test, y_test

In [ ]:
# Free the sampler (holds the full MadMiner file in memory)
del sampler
import gc; gc.collect()

## 2. Train CARL

CARL uses only the cross-entropy loss between the class label $y$ and the
classifier output $\hat{s}(x|\theta)$. No `r_xz` or `t_xz` needed.

In [ ]:
estimator = ParameterizedRatioEstimator(
    n_hidden=(200, 200, 200),
    activation="tanh",
)

losses_train, losses_val = estimator.train(
    method="carl",
    x="data/samples_carl/x_train.npy",
    y="data/samples_carl/y_train.npy",
    theta="data/samples_carl/theta0_train.npy",
    n_epochs=50,
    batch_size=256,
    initial_lr=1e-3,
    final_lr=1e-5,
    validation_split=0.25,
    early_stopping=True,
    early_stopping_patience=10,
)

estimator.save("models/carl")
print("Model saved to models/carl")

## 3. Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_train, label="Train")
ax.plot(losses_val, label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("CARL Training")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 4. Evaluate

In [ ]:
model = ParameterizedRatioEstimator()
model.load("models/carl")

x_test_arr = np.load("data/samples_carl/x_test.npy")
theta_test_arr = np.load("data/samples_carl/theta0_test.npy")

batch_size = 10_000
log_r_parts = []
for start in range(0, len(x_test_arr), batch_size):
    end = min(start + batch_size, len(x_test_arr))
    lr, _ = model.evaluate_log_likelihood_ratio(
        x=x_test_arr[start:end],
        theta=theta_test_arr[start:end],
        test_all_combinations=False,
    )
    log_r_parts.append(lr)
log_r_hat = np.concatenate(log_r_parts)

log_r_true = np.log(r_xz_test)

print(f"Predicted log r: shape={log_r_hat.shape}, mean={log_r_hat.mean():.3f}")
print(f"True log r:      shape={log_r_true.shape}, mean={log_r_true.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
finite = np.isfinite(log_r_true) & np.isfinite(log_r_hat)
ax.scatter(log_r_true[finite], log_r_hat[finite], alpha=0.02, s=2, rasterized=True)
lims = [max(log_r_true[finite].min(), -20), min(log_r_true[finite].max(), 20)]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel(r"True $\log\, r(x|\theta)$")
ax.set_ylabel(r"Predicted $\log\, \hat{r}(x|\theta)$")
ax.set_title("CARL: Predicted vs True")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax = axes[1]
residuals = log_r_hat[finite] - log_r_true[finite]
ax.hist(residuals, bins=100, range=(-5, 5), density=True, alpha=0.7)
ax.axvline(0, color="r", ls="--")
ax.set_xlabel(r"$\log\,\hat{r} - \log\,r$")
ax.set_ylabel("Density")
ax.set_title(f"Residuals (mean={residuals.mean():.3f}, std={residuals.std():.3f})")

plt.tight_layout()
plt.show()

## 5. Approximate NLL Scan

In [ ]:
# Load SM-distributed test events
x_test_arr = np.load("data/samples_carl/x_test.npy")
y_test_arr = np.load("data/samples_carl/y_test.npy")
sm_mask = (y_test_arr.flatten() == 1)
x_sm = x_test_arr[sm_mask][:2000]
del x_test_arr, y_test_arr

n_grid = 25
cwl2_vals = np.linspace(-20, 20, n_grid)
cpwl2_vals = np.linspace(-20, 20, n_grid)
cwl2_grid, cpwl2_grid = np.meshgrid(cwl2_vals, cpwl2_vals)
theta_grid = np.column_stack([cwl2_grid.ravel(), cpwl2_grid.ravel()])

# Evaluate in batches over theta grid
log_r_all = []
for i in range(len(theta_grid)):
    lr, _ = model.evaluate_log_likelihood_ratio(
        x=x_sm,
        theta=theta_grid[i:i+1],
        test_all_combinations=True,
    )
    log_r_all.append(lr.flatten())
log_r_all = np.array(log_r_all)  # (n_theta, n_x)

nll = -2.0 * log_r_all.sum(axis=1).reshape(n_grid, n_grid)
nll -= nll.min()

fig, ax = plt.subplots(figsize=(7, 6))
c = ax.contourf(cwl2_grid, cpwl2_grid, nll, levels=30, cmap="viridis")
plt.colorbar(c, ax=ax, label=r"$-2 \sum_i \log\, \hat{r}(x_i|\theta)$")
ax.contour(cwl2_grid, cpwl2_grid, nll, levels=[2.30, 5.99], colors=["white", "lightgray"], linewidths=2)
ax.plot(0, 0, "w*", markersize=15, label="SM")
ax.set_xlabel(r"$C_{W}/\Lambda^2$")
ax.set_ylabel(r"$C_{\tilde{W}}/\Lambda^2$")
ax.set_title("CARL: Approx. NLL")
ax.legend()
plt.tight_layout()
plt.show()